In [1]:
print("Distributed_DL")

Distributed_DL


In [2]:
import sys
print(sys.version)

3.10.20 (main, Mar  3 2026, 09:24:47) [GCC 13.3.0]


In [3]:
try:
    spark.stop()
except:
    pass

In [4]:
from pyspark.sql import SparkSession
spark= SparkSession.builder\
       .appName("Distributed_ML")\
       .getOrCreate()
spark
       
  


In [5]:
distributed_df=spark.read.parquet("/home/ubuntu/thesis/data/processed_data/system_state/distributed_data")
distributed_df.show(10)
distributed_df.count()                                                                                                           

+--------------------+----------+-----+----------+----------+-----+-----+-----+---------+-------+--------------------+-----------+
|                Page|      Date|value|Moving_avg|Moving_std|lag_1|lag_2|trend|deviation|z_score|          percentile|state_label|
+--------------------+----------+-----+----------+----------+-----+-----+-----+---------+-------+--------------------+-----------+
|10._August_de.wik...|2015-07-01|   33|      33.0|      NULL| NULL| NULL| NULL|      0.0|   NULL|                 0.0|     Normal|
|10._August_de.wik...|2016-03-10|   28|      28.0|       0.0|   28|   28|  0.0|      0.0|   NULL|                 0.0|     Normal|
|10._August_de.wik...|2016-07-22|   34|      34.0|       0.0|   34|   34|  0.0|      0.0|   NULL|                 0.0|     Normal|
|10._August_de.wik...|2016-12-05|   24|     24.67|      0.58|   25|   25| -1.0|    -0.67|  -1.16|0.005565862708719...|     Normal|
|10._August_de.wik...|2016-12-24|   15|      17.0|      1.73|   18|   18| -3.0|    

57168842

In [6]:
#distributed_df.coalesce(1).write.mode("overwrite").option("header", "true").csv("/home/ubuntu/thesis/distributed_csv")

In [7]:
from pyspark.sql.functions import col

data = distributed_df \
    .withColumn("value", col("value").cast("double")) \
    .withColumn("lag_1", col("lag_1").cast("double")) \
    .withColumn("lag_2", col("lag_2").cast("double"))

In [8]:
from pyspark.ml.feature import StringIndexer

page_indexer = StringIndexer(inputCol="Page", outputCol="Page_index")
data = page_indexer.fit(data).transform(data)

In [9]:
from pyspark.sql.functions import year, month, dayofmonth

data = data \
    .withColumn("year", year("Date")) \
    .withColumn("month", month("Date")) \
    .withColumn("day", dayofmonth("Date"))

In [10]:
feature_cols = ["value", "lag_1", "lag_2", "Page_index", "year", "month", "day"]

In [11]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
data = assembler.transform(data)

In [12]:
data.printSchema()

root
 |-- Page: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- value: double (nullable = true)
 |-- Moving_avg: double (nullable = true)
 |-- Moving_std: double (nullable = true)
 |-- lag_1: double (nullable = true)
 |-- lag_2: double (nullable = true)
 |-- trend: double (nullable = true)
 |-- deviation: double (nullable = true)
 |-- z_score: double (nullable = true)
 |-- percentile: double (nullable = true)
 |-- state_label: string (nullable = true)
 |-- Page_index: double (nullable = false)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- features: vector (nullable = true)



In [13]:
data = data.drop("features")

In [14]:
from pyspark.sql.functions import col, when, count

data.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in data.columns
]).show()

+----+----+-----+----------+----------+-----+------+-----+---------+-------+----------+-----------+----------+----+-----+---+
|Page|Date|value|Moving_avg|Moving_std|lag_1| lag_2|trend|deviation|z_score|percentile|state_label|Page_index|year|month|day|
+----+----+-----+----------+----------+-----+------+-----+---------+-------+----------+-----------+----------+----+-----+---+
|   0|   0|    0|         0|     92168|92168|183964|92168|        0| 114257|         0|          0|         0|   0|    0|  0|
+----+----+-----+----------+----------+-----+------+-----+---------+-------+----------+-----------+----------+----+-----+---+



In [15]:
data_clean= data.dropna()

In [16]:
from pyspark.sql.functions import col, when, count

data_clean.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in data_clean.columns
]).show()

26/05/05 15:47:06 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
                                                                                

+----+----+-----+----------+----------+-----+-----+-----+---------+-------+----------+-----------+----------+----+-----+---+
|Page|Date|value|Moving_avg|Moving_std|lag_1|lag_2|trend|deviation|z_score|percentile|state_label|Page_index|year|month|day|
+----+----+-----+----------+----------+-----+-----+-----+---------+-------+----------+-----------+----------+----+-----+---+
|   0|   0|    0|         0|         0|    0|    0|    0|        0|      0|         0|          0|         0|   0|    0|  0|
+----+----+-----+----------+----------+-----+-----+-----+---------+-------+----------+-----------+----------+----+-----+---+



In [17]:
data_clean.groupBy("state_label").count().show()

26/05/05 15:52:02 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 16:00:07 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
                                                                                

+-----------+--------+
|state_label|   count|
+-----------+--------+
|    Failure|14130716|
|   Degraded|19977434|
|     Normal|22855838|
+-----------+--------+



In [18]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
data_clean = assembler.transform(data_clean)

In [19]:
# # ============================================
# # SAVE SAMPLE DATASETS AS PARQUET
# # ============================================

# sample_1m = data_clean.limit(1000000)
# sample_3m = data_clean.limit(3000000)
# sample_5m = data_clean.limit(5000000)

# # Save datasets
# sample_1m.write.mode("overwrite").parquet("/home/ubuntu/thesis/sample_1m_parquet")
# sample_3m.write.mode("overwrite").parquet("/home/ubuntu/thesis/sample_3m_parquet")
# sample_5m.write.mode("overwrite").parquet("/home/ubuntu/thesis/sample_5m_parquet")

# print("Parquet datasets saved successfully.")

In [20]:
# ============================
# STEP 1: Create sampled datasets
# ============================

sample_datasets = {
    "1M": data_clean.limit(1000000),
    "3M": data_clean.limit(3000000),
    "5M": data_clean.limit(5000000)
}


# ============================
# STEP 2: Import required libraries
# ============================

from pyspark.ml.feature import StringIndexer, StandardScaler
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pandas as pd
import time


# ============================
# STEP 3: Initialize results list
# ============================

results = []


# ============================
# STEP 4: Loop through each sample
# ============================

for sample_name, sample_df in sample_datasets.items():

    print(f"\nProcessing ANN for sample size: {sample_name}")

    # Label Encoding
    label_indexer = StringIndexer(
        inputCol="state_label",
        outputCol="label"
    )

    indexed_df = label_indexer.fit(sample_df).transform(sample_df)

    # Feature Scaling
    scaler = StandardScaler(
        inputCol="features",
        outputCol="scaled_features",
        withMean=True,
        withStd=True
    )

    scaler_model = scaler.fit(indexed_df)
    scaled_df = scaler_model.transform(indexed_df)

    # Train-test split
    train, test = scaled_df.randomSplit([0.8, 0.2], seed=42)

    # Determine input/output layers
    input_size = len(feature_cols)
    num_classes = indexed_df.select("label").distinct().count()

    # ANN structure
    layers = [input_size, 64, 32, num_classes]

    # Define ANN model
    mlp = MultilayerPerceptronClassifier(
        layers=layers,
        labelCol="label",
        featuresCol="scaled_features",
        maxIter=100,
        blockSize=128,
        seed=42
    )

    # Train model and measure time
    start_time = time.time()

    model = mlp.fit(train)

    end_time = time.time()

    training_time = end_time - start_time

    # Predictions
    predictions = model.transform(test)

    # Evaluators
    accuracy_eval = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="accuracy"
    )

    precision_eval = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="weightedPrecision"
    )

    recall_eval = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="weightedRecall"
    )

    f1_eval = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="f1"
    )

    # Calculate metrics
    accuracy = accuracy_eval.evaluate(predictions)
    precision = precision_eval.evaluate(predictions)
    recall = recall_eval.evaluate(predictions)
    f1 = f1_eval.evaluate(predictions)

    # Save results
    results.append({
        "Sample_Size": sample_name,
        "Rows": sample_df.count(),
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1_Score": f1,
        "Training_Time_Seconds": training_time
    })

    # Print metrics
    print(f"Accuracy: {accuracy}")
    print(f"Precision: {precision}")
    print(f"Recall: {recall}")
    print(f"F1 Score: {f1}")
    print(f"Training Time (seconds): {training_time}")


# ============================
# STEP 5: Final results
# ============================

results_df = pd.DataFrame(results)

print("\nANN Scalability Results:")
print(results_df)


# ============================
# STEP 6: Save results
# ============================

results_df.to_csv("/home/ubuntu/thesis/ann_scalability_results.csv", index=False)

print("\nResults saved successfully.")


Processing ANN for sample size: 1M


26/05/05 16:01:12 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 16:05:10 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 16:05:29 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 16:10:38 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 16:10:59 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 16:13:50 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 16:13:56 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 16:15:04 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 16:15:57 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 16:16:09 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 16:16:10 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/05/05 16:16:19 WARN 

Accuracy: 0.6009407309449387
Precision: 0.6286933590923893
Recall: 0.6009407309449387
F1 Score: 0.5990372720427295
Training Time (seconds): 1037.1608095169067

Processing ANN for sample size: 3M


26/05/05 16:35:11 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 16:35:55 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 16:35:59 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 16:37:25 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 16:37:30 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 16:38:17 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 16:38:20 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 16:40:51 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 16:43:33 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 16:44:00 WARN MemoryStore: Not enough space to cache rdd_318_0 in memory! (computed 177.1 MiB so far)
26/05/05 16:44:00 WARN BlockManager: Persisting block rdd_318_0 to disk instead.
26/05/05 16:44:04 WARN DA

Accuracy: 0.6076735868763108
Precision: 0.6375324400511155
Recall: 0.6076735868763108
F1 Score: 0.6040960021912104
Training Time (seconds): 2944.586109161377

Processing ANN for sample size: 5M


26/05/05 17:38:28 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 17:39:40 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 17:39:45 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 17:41:57 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 17:42:04 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 17:43:14 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 17:43:18 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 17:48:08 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 17:51:45 WARN DAGScheduler: Broadcasting large task binary with size 11.2 MiB
26/05/05 17:52:32 WARN MemoryStore: Not enough space to cache rdd_546_0 in memory! (computed 114.0 MiB so far)
26/05/05 17:52:32 WARN BlockManager: Persisting block rdd_546_0 to disk instead.
26/05/05 17:52:43 WARN DA

Accuracy: 0.5015565039450446
Precision: 0.517511122878076
Recall: 0.5015565039450445
F1 Score: 0.47228121838921405
Training Time (seconds): 4918.750368356705

ANN Scalability Results:
  Sample_Size     Rows  Accuracy  Precision    Recall  F1_Score  \
0          1M  1000000  0.600941   0.628693  0.600941  0.599037   
1          3M  3000000  0.607674   0.637532  0.607674  0.604096   
2          5M  5000000  0.501557   0.517511  0.501557  0.472281   

   Training_Time_Seconds  
0            1037.160810  
1            2944.586109  
2            4918.750368  

Results saved successfully.


In [21]:
#The experimental results indicate that distributed deep learning architectures should be regarded as scalable computational frameworks rather than superior predictive alternatives to centralized systems. 
#While centralized models achieve significantly better classification performance,distributed systems provide critical advantages in scalability, resource utilization, and enterprise-level deployment.

In [22]:
#To evaluate deep learning scalability in distributed environments, ANN was initially implemented across sampled distributed datasets (1M, 3M, and 5M).
#The ANN results demonstrated positive scalability from 1M to 3M samples; however, performance degradation at 5M indicated scalability saturation,
#likely due to computational bottlenecks, architectural limitations, or increased distributed overhead. 
#To further validate deep learning performance under distributed large-scale conditions, GRU was introduced as an alternative recurrent
#deep learning architecture. Given the increased computational demands of GRU, Google Colab was utilized to provide enhanced computational 
#resources for sampled distributed training and evaluation.”                        